# Feature Engineering and Modelling

---

1. Import packages
2. Load data
3. Modelling

---

## 1. Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

%matplotlib inline
sns.set(color_codes=True)

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

## 2. Load data

In [ ]:
df = pd.read_csv('./data_for_predictions.csv')
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

In [ ]:
print(f'Dataset shape : {df.shape}')
print(f'Churn rate    : {df["churn"].mean()*100:.2f}%')
print()
print('Class distribution:')
print(df['churn'].value_counts())
churn_ratio = df['churn'].value_counts()[0] / df['churn'].value_counts()[1]
print(f'\nClass imbalance ratio: {churn_ratio:.1f}:1 (Not Churned : Churned)')
print('=> We will use class_weight="balanced" to handle this imbalance.')

## 3. Modelling

We now have a dataset containing features that we have engineered and we are ready to start training a predictive model. We will focus on training a **Random Forest** classifier.

### Why Random Forest?

A Random Forest is an **ensemble** method that builds a collection (forest) of Decision Trees. It is powerful because:
- Many weak learners (trees) together form a highly predictive model
- Uses averaging/voting across all trees — more robust than a single tree
- Does **not** require feature scaling (rule-based, not distance-based)
- Handles non-linear relationships well
- Provides feature importance scores, which help validate our price sensitivity hypothesis

### Data sampling

We split our dataset into training (75%) and test (25%) samples. We use **stratified splitting** (`stratify=y`) to ensure both splits maintain the same churn ratio, which is critical for imbalanced datasets.

In [ ]:
train_df = df.copy()

y = df['churn']
X = df.drop(columns=['id', 'churn'])
print(X.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(f'\nTrain churn rate: {y_train.mean()*100:.2f}%')
print(f'Test  churn rate: {y_test.mean()*100:.2f}%')

### Model training

We configure the Random Forest with the following key parameters:

| Parameter | Value | Reason |
|-----------|-------|--------|
| `n_estimators` | 1000 | More trees = more stable predictions |
| `max_depth` | 8 | Limits depth to avoid overfitting |
| `min_samples_leaf` | 10 | Prevents learning noise from tiny subgroups |
| `class_weight` | 'balanced' | Compensates for the 9.3:1 class imbalance |
| `random_state` | 42 | Ensures reproducibility |

In [ ]:
model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=8,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print('Model trained successfully!')

### Evaluation

In [ ]:
# Generate predictions
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(f'Predictions generated for {len(y_pred)} test samples.')
print(f'Predicted churned : {y_pred.sum()} ({y_pred.mean()*100:.2f}%)')
print(f'Actual churned    : {y_test.sum()} ({y_test.mean()*100:.2f}%)')

In [ ]:
# Calculate performance metrics
accuracy  = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred, zero_division=0)
recall    = metrics.recall_score(y_test, y_pred, zero_division=0)
f1        = metrics.f1_score(y_test, y_pred, zero_division=0)
roc_auc   = metrics.roc_auc_score(y_test, y_pred_proba)

print('=== MODEL PERFORMANCE METRICS ===')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print()
print(metrics.classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned'], zero_division=0))

#### Why did we choose these evaluation metrics?

**1. Accuracy** is reported but is not our primary metric. On an imbalanced dataset (only ~9.7% churn), a naive model that always predicts "Not Churned" would achieve ~90% accuracy — yet it would be completely useless. Accuracy is misleading here.

**2. Precision** — Of all customers we *predicted* to churn, how many actually did? High precision reduces wasted retention spend on customers who were not going to leave.

**3. Recall** — Of all customers who *actually* churned, how many did we catch? High recall is critical — missing a churning customer means permanently losing that revenue. In a churn context, this is often the more important of the two.

**4. F1 Score** — The harmonic mean of Precision and Recall. This is our **primary metric** because it balances both concerns and is robust to class imbalance. It penalises models that are extreme in either direction.

**5. ROC-AUC** — Measures discriminative ability across all thresholds. Score above 0.5 is better than random; 1.0 is perfect. This is threshold-independent and robust to imbalance — ideal for comparing models.

In [ ]:
# Confusion Matrix
cm = metrics.confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Pred: Not Churned', 'Pred: Churned'],
    yticklabels=['Actual: Not Churned', 'Actual: Churned']
)
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = metrics.roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'gray', linestyle='--', label='Random (AUC = 0.50)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

plt.suptitle('Model Evaluation — Random Forest Churn Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correct non-churn) : {tn}')
print(f'False Positives (incorrect flag)     : {fp}')
print(f'False Negatives (missed churners)    : {fn}')
print(f'True Positives  (correctly caught)   : {tp}')

In [ ]:
# Feature Importance Plot
feature_importance = pd.Series(
    model.feature_importances_, index=X.columns
).sort_values(ascending=False)

top_n = 15
top_features = feature_importance.head(top_n)

plt.figure(figsize=(10, 7))
colors = ['#2c7bb6' if 'price' in c or 'peak' in c or 'offpeak' in c else '#d7191c'
          if 'margin' in c else '#1a9641' for c in top_features.index]
top_features[::-1].plot(kind='barh', color=colors[::-1], edgecolor='white')
plt.title(f'Top {top_n} Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d7191c', label='Margin features'),
    Patch(facecolor='#2c7bb6', label='Price features'),
    Patch(facecolor='#1a9641', label='Other features')
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Top {top_n} most important features:')
print(top_features.round(4))

### Is the model performance satisfactory?

**Summary of results:**

| Metric | Score | Assessment |
|--------|-------|------------|
| Accuracy | ~78% | Misleading due to class imbalance — not primary metric |
| Precision | ~20% | Low — model flags some non-churners incorrectly |
| Recall | ~44% | Moderate — we catch ~44% of actual churners |
| F1 Score | ~28% | Below ideal — room for improvement |
| ROC-AUC | ~0.70 | Meaningfully better than random (0.50) |

**Justification:**

The model performance is **partially satisfactory** for a first iteration. The ROC-AUC of ~0.70 demonstrates that the model has genuine discriminative ability — far beyond random guessing. A recall of ~44% already means we can proactively identify nearly half of all churning customers and target them with retention campaigns, which delivers real business value.

The lower precision (~20%) and F1 (~28%) are expected given the severe class imbalance (9.3:1). We addressed this using `class_weight='balanced'`, which substantially improved recall compared to an unweighted model (which achieved 0% recall entirely).

**Key business insight from feature importances:**

Margin metrics (`margin_net_pow_ele`, `margin_gross_pow_ele`) and activity features (`months_activ`, `cons_12m`) are the top predictors. While price-related features do appear in the top 15 (e.g. `off_peak_peak_var_mean_diff`, `var_year_price_off_peak`), they are **not the dominant drivers**. This **partially challenges the initial hypothesis** — price sensitivity matters, but customer profitability and engagement patterns are equally or more predictive of churn. This finding should be communicated clearly to the client.

**Next steps for improvement:**
1. Threshold tuning — lower the decision threshold to increase recall at acceptable precision cost
2. SMOTE oversampling during training for better class balance
3. Hyperparameter tuning via `GridSearchCV` or `RandomizedSearchCV`
4. Test gradient boosting (XGBoost, LightGBM) which typically outperform RF on tabular imbalanced data
5. StratifiedKFold cross-validation for more robust metric estimates

In [ ]:
# Save predictions output
results_df = X_test.copy()
results_df['actual_churn']      = y_test.values
results_df['predicted_churn']   = y_pred
results_df['churn_probability'] = y_pred_proba

results_df.to_csv('churn_predictions.csv', index=False)
print('Predictions saved to churn_predictions.csv')
print(f'Total test records : {len(results_df)}')
print(f'Predicted churners : {y_pred.sum()}')